# Legal RAG Assistant — Backend

This notebook implements the complete Retrieval-Augmented Generation (RAG) backend for the Indian Startup Legal Assistant.

### Architecture Overview
1. **Environment and Configuration**: Dependency loading, Google Drive mount, and retrieval hyperparameters.
2. **Load Retrieval Artifacts**: Loading FAISS vector index, BM25 index, corpus, and `all-MiniLM-L6-v2` embedding model.
3. **Hybrid Retrieval Engine**: Semantic search, BM25 search, parent-section deduplication, Weighted Reciprocal Rank Fusion (RRF), and metadata filtering.
4. **LLM Integration**: Google GenAI / Gemma client configuration and robust JSON response parsing.
5. **Query Understanding**: Structured intent analysis, legal domain classification, entity extraction, and ambiguity/clarification detection.
6. **Answer Synthesis and Citation Grounding**: Grounded legal response generation strictly constrained to retrieved evidence.
7. **Verification**: Deterministic citation integrity validation and evidential abstention checks.
8. **LangGraph Workflow**: State machine with a conditional clarification branch orchestrating understanding, clarification routing, hybrid retrieval, synthesis, and verification.
9. **End-to-End Testing**: Representative end-to-end execution of the legal assistant pipeline.
10. **Evaluation**: Preliminary evaluation on startup-specific legal inquiries.

## 1. Environment and Configuration

Install dependencies, import required libraries, mount Google Drive, and set operational parameters.

In [ ]:
!pip install -q faiss-cpu sentence-transformers rank-bm25 langgraph langchain google-genai
!pip -q install gradio

In [ ]:
import os
import json
import pickle
import re
import numpy as np
import faiss

from typing import TypedDict, Optional, List, Dict, Any
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

from google import genai
from langgraph.graph import StateGraph, END
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Artifact directory
ARTIFACT_DIR = "/content/drive/MyDrive/Gemma DTU Outputs"

# Embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

# Retrieval configuration
CANDIDATE_K = 50
FINAL_TOP_K = 5

RRF_K = 60
SEMANTIC_WEIGHT = 2.0
KEYWORD_WEIGHT = 0.75

print("Configuration loaded.")

## 2. Load Retrieval Artifacts

Load precomputed FAISS vector index, BM25 lexical index, tokenized corpus, and chunked retrieval documents generated in Notebook 1.

In [ ]:
# Load embedding model
embedding_model = SentenceTransformer(
    os.path.join(ARTIFACT_DIR, EMBEDDING_MODEL_NAME)
)

# Load FAISS index
faiss_index = faiss.read_index(
    os.path.join(ARTIFACT_DIR, "chunked_faiss.index")
)

# Load chunked retrieval documents
with open(os.path.join(ARTIFACT_DIR, "chunked_retrieval_documents.pkl"), "rb") as f:
    retrieval_documents = pickle.load(f)

# Load BM25 index
with open(os.path.join(ARTIFACT_DIR, "chunked_bm25.pkl"), "rb") as f:
    bm25_index = pickle.load(f)

# Load BM25 corpus
with open(os.path.join(ARTIFACT_DIR, "chunked_bm25_corpus.pkl"), "rb") as f:
    bm25_corpus = pickle.load(f)

print("All retrieval artifacts loaded successfully.")

In [ ]:
print("FAISS vectors      :", faiss_index.ntotal)
print("Retrieval documents:", len(retrieval_documents))
print("BM25 corpus entries:", len(bm25_corpus))
print("Embedding model    :", EMBEDDING_MODEL_NAME)

assert faiss_index.ntotal == len(retrieval_documents)
assert len(bm25_corpus) == len(retrieval_documents)

print("\nArtifact validation passed.")

In [ ]:
from collections import Counter, defaultdict
import random

# 1. chunk_id uniqueness
chunk_ids = [d["chunk_id"] for d in retrieval_documents]
dups = [k for k, v in Counter(chunk_ids).items() if v > 1]
print(f"Unique chunk_ids: {len(set(chunk_ids))} / {len(chunk_ids)} | duplicated IDs: {len(dups)}")

# 2. Parent IDs shared by different sections
variants = defaultdict(set)
for d in retrieval_documents:
    m = d["metadata"]
    variants[d["parent_chunk_id"]].add((m.get("chapter"), m.get("section_title"), m.get("source_file")))
collisions = {p: v for p, v in variants.items() if len(v) > 1}
print("Parent IDs shared by different sections:", len(collisions))
for p, v in list(collisions.items())[:5]:
    print("  ", p, "->", list(v)[:3])

# 3. Embedding truncation check
tok = embedding_model.tokenizer
limit = embedding_model.max_seq_length
sample = random.Random(0).sample(retrieval_documents, 2000)
lens = [len(tok(d["embedding_text"])["input_ids"]) for d in sample]
over = sum(l > limit for l in lens)
print(f"max_seq_length={limit} | median tokens={int(np.median(lens))} | over limit: {over}/{len(lens)} ({100*over/len(lens):.1f}%)")

# 4. Which models does your key actually see?
for m in client.models.list():
    if "gemini" in m.name:
        print(m.name)

## 3. Hybrid Retrieval Engine

Implement dense semantic search, sparse BM25 search, parent-section deduplication, Weighted Reciprocal Rank Fusion (RRF), and metadata-based statutory filtering.

In [ ]:
# 1. Semantic Search
def semantic_search(query, top_k=10):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = np.asarray(query_embedding, dtype=np.float32)

    scores, indices = faiss_index.search(
        query_embedding,
        min(top_k, faiss_index.ntotal)
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx < 0:
            continue

        doc = retrieval_documents[idx]

        results.append({
            "score": float(score),
            "chunk_id": doc["chunk_id"],
            "parent_chunk_id": doc["parent_chunk_id"],
            "metadata": doc["metadata"],
            "text": doc["text"]
        })

    return results


# 2. BM25 Keyword Search
def bm25_search(query, top_k=10):

    tokens = re.findall(r"\w+", query.lower())
    scores = bm25_index.get_scores(tokens)

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    results = []

    for idx, score in ranked:

        doc = retrieval_documents[idx]

        results.append({
            "score": float(score),
            "chunk_id": doc["chunk_id"],
            "parent_chunk_id": doc["parent_chunk_id"],
            "metadata": doc["metadata"],
            "text": doc["text"]
        })

    return results


# 3. Deduplication by Parent Section
def deduplicate_ranking(ranking):

    seen = set()
    unique_results = []

    for result in ranking:

        parent_id = result["parent_chunk_id"]

        if parent_id not in seen:
            unique_results.append(result)
            seen.add(parent_id)

    return unique_results


# 4. Weighted Reciprocal Rank Fusion
def reciprocal_rank_fusion(
    semantic_ranking,
    keyword_ranking,
    k=RRF_K,
    semantic_weight=SEMANTIC_WEIGHT,
    keyword_weight=KEYWORD_WEIGHT
):

    fused_scores = {}
    document_lookup = {}

    rankings = [
        (semantic_ranking, semantic_weight),
        (keyword_ranking, keyword_weight)
    ]

    for ranking, weight in rankings:

        for rank, result in enumerate(ranking):

            parent_id = result["parent_chunk_id"]

            document_lookup.setdefault(parent_id, result)

            fused_scores[parent_id] = (
                fused_scores.get(parent_id, 0)
                + weight / (k + rank + 1)
            )

    return [
        {
            **document_lookup[parent_id],
            "rrf_score": score
        }
        for parent_id, score in sorted(
            fused_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]


# 5. Metadata Filter Extraction
def extract_metadata_filters(query):

    filters = {}
    q = query.lower()

    match = re.search(r"section\s+([0-9a-z]+)", q)

    if match:
        filters["section_number"] = match.group(1)

    for doc in retrieval_documents:

        act = doc["metadata"].get("act_name")

        if act and act.lower().replace(",", "") in q:
            filters["act_name"] = act
            break

    return filters


# 6. Complete Hybrid Search
def hybrid_search(query, top_k=FINAL_TOP_K):

    semantic_results = deduplicate_ranking(
        semantic_search(query, CANDIDATE_K)
    )

    keyword_results = deduplicate_ranking(
        bm25_search(query, CANDIDATE_K)
    )

    fused = reciprocal_rank_fusion(
        semantic_results,
        keyword_results
    )

    filters = extract_metadata_filters(query)

    if filters:

        filtered = [
            result for result in fused
            if all(
                str(result["metadata"].get(key, "")).lower()
                == str(value).lower()
                for key, value in filters.items()
            )
        ]

        if filtered:
            fused = filtered

    return fused[:top_k]


print("Hybrid retrieval engine initialized.")

## 4. LLM Integration

Initialize the Google GenAI client securely using Colab Secrets, and define generation and JSON response parsing functions.

In [ ]:
from google.colab import userdata
from google.genai import types

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in Colab Secrets.")

# timeout is in milliseconds: no single request may hang longer than 30 s
client = genai.Client(
    api_key=GOOGLE_API_KEY,
    http_options=types.HttpOptions(timeout=30_000),
)

print("Gemini client initialized.")

In [ ]:
import time
import random
import httpx
from google.genai import types
from google.genai.errors import APIError

# Model priority lists (first = preferred). Edit these after running the probe cell.
UNDERSTAND_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
SYNTHESIS_MODELS = ["gemini-3.5-flash", "gemini-3.1-flash-lite", "gemini-2.5-flash"]

RETRYABLE_CODES = {500, 502, 503, 504}   # retried on the same model with backoff
MAX_ATTEMPTS_PER_MODEL = 2
BACKOFF_BASE_SECONDS = 2.0
TOTAL_DEADLINE_SECONDS = 90              # hard cap for one llm_generate call


class LLMUnavailableError(RuntimeError):
    """All configured models failed or the deadline was exceeded."""


def llm_generate(prompt, models, temperature=0.1, max_tokens=4096, json_mode=True):
    config = types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=max_tokens,
        response_mime_type="application/json" if json_mode else None,
    )

    start = time.time()
    failures = []

    for model in models:
        for attempt in range(1, MAX_ATTEMPTS_PER_MODEL + 1):

            if time.time() - start > TOTAL_DEADLINE_SECONDS:
                raise LLMUnavailableError(
                    f"Deadline of {TOTAL_DEADLINE_SECONDS}s exceeded. Failures: {failures}"
                )

            try:
                response = client.models.generate_content(
                    model=model, contents=prompt, config=config
                )
                text = response.text
                if not text or not text.strip():
                    raise ValueError("empty response")

                print(f"[LLM] {model} succeeded in {time.time() - start:.1f}s")
                return text

            except APIError as e:
                code = getattr(e, "code", None)

                if code == 404:
                    print(f"[LLM] {model}: 404 model unavailable, next model.")
                    failures.append((model, 404))
                    break

                if code == 429:
                    print(f"[LLM] {model}: 429 quota/rate limit, next model.")
                    failures.append((model, 429))
                    break

                if code not in RETRYABLE_CODES:
                    raise   # 400/401/403 etc: a config problem, do not mask it

                reason = f"HTTP {code}"

            except (httpx.TimeoutException, httpx.TransportError):
                reason = "timeout/network error"

            except ValueError as e:
                if str(e) != "empty response":
                    raise
                reason = "empty response"

            failures.append((model, reason))
            print(f"[LLM] {model}: {reason} (attempt {attempt}/{MAX_ATTEMPTS_PER_MODEL})")

            if attempt < MAX_ATTEMPTS_PER_MODEL:
                wait = BACKOFF_BASE_SECONDS * 2 ** (attempt - 1) + random.uniform(0, 1)
                print(f"[LLM] retrying in {wait:.1f}s")
                time.sleep(wait)

    raise LLMUnavailableError(f"All models failed: {failures}")


def parse_json_response(response):

    if not response:
        raise ValueError("Empty LLM response.")

    cleaned = response.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No valid JSON object found.")

    return json.loads(cleaned[start:end + 1])


print("LLM integration with fallback ready.")

## 5. Query Understanding

Extract language, legal intent, domain category, entities, and detect whether clarification is necessary before executing search.

In [ ]:
UNDERSTAND_PROMPT = """
Analyze this legal query.

Return ONLY valid JSON.

Schema:

{{
    "language": "",
    "intent": "",
    "domain": "",
    "entities": {{}},
    "confidence": 0.0,
    "needs_clarification": false,
    "clarification_question": ""
}}

Rules:
- Identify the language of the query.
- Identify the user's intent.
- Identify the relevant legal domain.
- Extract relevant entities.
- Set confidence between 0 and 1.
- If the query is ambiguous, set needs_clarification to true
  and provide one clarification question.
- Otherwise, set needs_clarification to false.

USER QUERY:
{query}
"""

In [ ]:
def understand_query(query):

    prompt = UNDERSTAND_PROMPT.format(query=query)

    response = llm_generate(
        prompt,
        UNDERSTAND_MODELS,
        temperature=0.1,
        max_tokens=4096
    )

    return parse_json_response(response)


print("Query understanding initialized.")

## 6. Answer Synthesis and Citation Grounding

Synthesize legal advice strictly grounded in retrieved statutory text with mandatory citation mapping and limitation reporting.

In [ ]:
SYNTHESIZE_PROMPT = """
You are an AI legal assistant for Indian citizens.

Answer the user's question using ONLY the retrieved legal evidence.

STRICT RULES:
1. Do not invent laws, sections, rights, penalties or procedures.
2. Every legal claim must be supported by the supplied evidence.
3. Cite only evidence IDs that directly support the claim.
4. If evidence is insufficient, clearly state the limitation.
5. Explain the law in simple, practical language.
6. Distinguish legal provisions from practical suggestions.
7. Do not assume facts that the user has not provided.
8. Do not present this answer as a substitute for professional legal advice.
9. Do not describe an action as mandatory unless the legal text uses "shall"
   or "must"; if it says "may", say it is optional.

Return ONLY valid JSON using this schema:

{{
    "summary": "Direct answer to the question",
    "rights": [],
    "options": [],
    "next_steps": [],
    "used_sources": [],
    "insufficient_evidence": false,
    "disclaimer": "This is general legal information and not a substitute for professional legal advice."
}}

IMPORTANT:
- used_sources must contain exact evidence IDs from the supplied evidence.
- Do not invent or modify evidence IDs.
- If no evidence supports the answer, return an empty used_sources list.
- Set insufficient_evidence to true when the evidence cannot adequately answer the question.

USER QUESTION:
{query}

RETRIEVED LEGAL EVIDENCE:
{context}
"""

In [ ]:
def synthesize_answer(query, retrieved_chunks):

    context_parts = []

    for chunk in retrieved_chunks:

        metadata = chunk["metadata"]

        context_parts.append(f"""
EVIDENCE ID: {chunk["chunk_id"]}

Act: {metadata.get("act_name")}
Section: {metadata.get("section_number")}
Title: {metadata.get("section_title")}

Legal text:
{chunk["text"]}
""")

    context = "\n\n".join(context_parts)

    prompt = SYNTHESIZE_PROMPT.format(
        query=query,
        context=context
    )

    response = llm_generate(
        prompt,
        SYNTHESIS_MODELS,
        temperature=0.1,
        max_tokens=8192
    )

    result = parse_json_response(response)

    required_keys = [
        "summary", "rights", "options", "next_steps",
        "used_sources", "insufficient_evidence", "disclaimer"
    ]

    for key in required_keys:
        if key not in result:
            raise ValueError(f"Missing key in synthesis response: {key}")

    valid_ids = {chunk["chunk_id"] for chunk in retrieved_chunks}

    # Record whether the MODEL said the evidence was insufficient,
    # before our own override below.
    result["model_abstained"] = bool(result["insufficient_evidence"])

    # Keep only real, retrieved IDs (deduplicated, order preserved).
    result["used_sources"] = list(dict.fromkeys(
        s for s in result["used_sources"]
        if isinstance(s, str) and s in valid_ids
    ))

    if not result["used_sources"]:
        result["insufficient_evidence"] = True

    return result


print("Grounded synthesis function ready.")

## 7. Verification

Verify that citations link to valid retrieved chunks, enforce evidential abstentions, and ensure cited legal evidence contains actual text.

In [ ]:
def verify_answer(answer, citations, retrieved_chunks, insufficient_evidence):

    if not answer or not answer.strip():
        return False, ["Answer is empty."]

    retrieved_lookup = {
        chunk["chunk_id"]: chunk
        for chunk in retrieved_chunks
    }

    citation_ids = {citation["chunk_id"] for citation in citations}

    invalid_ids = citation_ids - set(retrieved_lookup.keys())

    if invalid_ids:
        return False, ["One or more citation IDs are invalid."]

    if not citation_ids:
        if insufficient_evidence:
            return True, [
                "Abstention: no supporting evidence was cited and the system "
                "flagged insufficient evidence. No uncited legal claims shown."
            ]
        return False, ["No citations and evidence was not flagged as insufficient."]

    for citation_id in citation_ids:
        if not retrieved_lookup[citation_id].get("text", "").strip():
            return False, [f"Cited evidence {citation_id} has no legal text."]

    return True, [
        "Citation integrity passed. Legal claim entailment has NOT been independently verified."
    ]


print("Deterministic verification function initialized.")

## 8. LangGraph Workflow

Define the typed workflow state, node handlers, conditional clarification router, and assemble the compiled execution graph.

In [ ]:
class LegalAssistantState(TypedDict):

    # User input
    query: str

    # Query understanding
    language: Optional[str]
    intent: Optional[str]
    domain: Optional[str]
    entities: Dict[str, Any]
    confidence: Optional[float]

    # Clarification
    needs_clarification: bool
    clarification_question: Optional[str]

    # Retrieval
    retrieved_chunks: List[Dict]

    # Synthesis
    answer: Optional[str]
    insufficient_evidence: bool
    disclaimer: Optional[str]

    # Verification
    citations: List[Dict]
    verified: bool
    verification_notes: List[str]


print("LegalAssistantState defined.")

In [ ]:
# 1. Understand Node
def understand_node(state: LegalAssistantState):

    result = understand_query(state["query"])

    return {
        "language": result.get("language"),
        "intent": result.get("intent"),
        "domain": result.get("domain"),
        "entities": result.get("entities", {}),
        "confidence": result.get("confidence"),
        "needs_clarification": result.get("needs_clarification", False),
        "clarification_question": result.get("clarification_question")
    }


# 2. Clarification Node
def clarify_node(state: LegalAssistantState):

    question = state.get("clarification_question")

    return {
        "answer": question or "Could you provide more details about your legal issue?",
        "citations": [],
        "verified": False,
        "verification_notes": ["Clarification required before retrieval."]
    }


# 3. Retrieval Node
def retrieve_node(state: LegalAssistantState):

    results = hybrid_search(
        state["query"],
        top_k=FINAL_TOP_K
    )

    return {
        "retrieved_chunks": results
    }


# 4. Synthesis Node
ABSTENTION_MESSAGE = (
    "I could not find sufficient evidence in the indexed legal texts "
    "to answer this reliably."
)
STANDARD_DISCLAIMER = (
    "This is general legal information and not a substitute for "
    "professional legal advice."
)


def synthesize_node(state: LegalAssistantState):

    result = synthesize_answer(
        state["query"],
        state["retrieved_chunks"]
    )

    valid_chunks = {
        chunk["chunk_id"]: chunk
        for chunk in state["retrieved_chunks"]
    }

    citations = []

    for source_id in result["used_sources"]:

        metadata = valid_chunks[source_id]["metadata"]

        citations.append({
            "chunk_id": source_id,
            "act_name": metadata.get("act_name"),
            "section_number": metadata.get("section_number"),
            "section_title": metadata.get("section_title")
        })

    if citations:

        answer_parts = [result["summary"]]

        for heading, key in [
            ("Rights and protections", "rights"),
            ("Available options", "options"),
            ("Suggested next steps", "next_steps")
        ]:
            if result.get(key):
                answer_parts.append(
                    f"\n{heading}:\n" +
                    "\n".join(f"- {item}" for item in result[key])
                )

    else:
        # No valid citation: never show uncited legal claims.
        if result["model_abstained"]:
            answer_parts = [result["summary"]]
        else:
            answer_parts = [ABSTENTION_MESSAGE]

        answer_parts.append(
            "\nSuggested next step:\n"
            "- Consult a qualified legal professional about this question."
        )

    answer_parts.append(f"\n{STANDARD_DISCLAIMER}")

    return {
        "answer": "\n".join(answer_parts),
        "citations": citations,
        "insufficient_evidence": result["insufficient_evidence"],
        "disclaimer": STANDARD_DISCLAIMER
    }
# 5. Verification Node
def verify_node(state: LegalAssistantState):

    verified, notes = verify_answer(
        answer=state["answer"],
        citations=state["citations"],
        retrieved_chunks=state["retrieved_chunks"],
        insufficient_evidence=state["insufficient_evidence"]
    )

    return {
        "verified": verified,
        "verification_notes": notes
    }


# 6. Router
def clarification_router(state: LegalAssistantState):

    if state.get("needs_clarification", False):
        return "clarify"

    return "retrieve"


print("All LangGraph nodes defined.")

In [ ]:
graph = StateGraph(LegalAssistantState)

# Register nodes
graph.add_node("understand", understand_node)
graph.add_node("clarify", clarify_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("synthesize", synthesize_node)
graph.add_node("verify", verify_node)

# Entry point
graph.set_entry_point("understand")

# Conditional routing
graph.add_conditional_edges(
    "understand",
    clarification_router,
    {
        "clarify": "clarify",
        "retrieve": "retrieve"
    }
)

# Main workflow
graph.add_edge("clarify", END)
graph.add_edge("retrieve", "synthesize")
graph.add_edge("synthesize", "verify")
graph.add_edge("verify", END)

# Compile
legal_graph = graph.compile()

print("Legal RAG LangGraph workflow compiled successfully.")

## 9. End-to-End Testing

Execute a representative end-to-end test query through the compiled LangGraph workflow to demonstrate understanding, hybrid retrieval, grounded synthesis, citations, and verification.

In [ ]:
# Representative end-to-end test query
test_query = "Can my employer fire me while I am pregnant?"

initial_state = {
    "query": test_query,
    "language": None,
    "intent": None,
    "domain": None,
    "entities": {},
    "confidence": None,
    "needs_clarification": False,
    "clarification_question": None,
    "retrieved_chunks": [],
    "answer": None,
    "insufficient_evidence": False,
    "disclaimer": None,
    "citations": [],
    "verified": False,
    "verification_notes": []
}

result = legal_graph.invoke(initial_state)

print("\n" + "=" * 60)
print("FINAL LEGAL ANSWER")
print("=" * 60)
print(result["answer"])

print("\n" + "=" * 60)
print("CITATIONS")
print("=" * 60)

for citation in result["citations"]:
    print(
        citation["act_name"],
        "| Section", citation["section_number"],
        "|", citation["section_title"],
        "| ID:", citation["chunk_id"]
    )

print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)

print("Verified             :", result["verified"])
print("Insufficient evidence:", result["insufficient_evidence"])
print("Verification notes   :", result["verification_notes"])

In [ ]:
from collections import Counter

# What does the corpus actually contain?
acts = Counter(d["metadata"]["act_name"] for d in retrieval_documents)
print(f"{len(acts)} acts. Chunks per act:")
for a, n in acts.most_common(60):
    print(f"{n:5d}  {a}")

# Is the copyright-ownership provision present, and where does it rank?
q = "Can I use a freelancer's work commercially without an agreement?"
hits = [d for d in retrieval_documents
        if "first owner of the copyright" in d["text"].lower()]
hit_parents = {d["parent_chunk_id"] for d in hits}
print("\nChunks containing 'first owner of the copyright':", len(hits))
for d in hits[:5]:
    print("  ", d["chunk_id"])

ranked = hybrid_search(q, top_k=200)
print("Fused candidates:", len(ranked))
for i, r in enumerate(ranked, 1):
    if r["parent_chunk_id"] in hit_parents:
        print("  ownership provision at rank", i, "->", r["chunk_id"])

# The rest of Section 68(2)
for d in retrieval_documents:
    if d["chunk_id"] == "The_Code_On_Social_Security_2020_68_part_002":
        print("\n", d["text"][:800])

## 10. Preliminary Evaluation

This section presents a preliminary qualitative evaluation of the assistant on a startup-specific commercial query regarding intellectual property and contract law.

**Evaluation**: `Small qualitative query set`

In [ ]:
def new_state(q):
    return {
        "query": q, "language": None, "intent": None, "domain": None,
        "entities": {}, "confidence": None, "needs_clarification": False,
        "clarification_question": None, "retrieved_chunks": [], "answer": None,
        "insufficient_evidence": False, "disclaimer": None, "citations": [],
        "verified": False, "verification_notes": [],
    }

EVAL_QUERIES = [
    "Can my employer fire me while I am pregnant?",
    "Can I use a freelancer's work commercially without an agreement?",
    "What registration is needed to start a private limited company?",
    "What are the duties of directors of a company?",
    "Is an employer required to pay gratuity?",
    "How can I protect my startup's brand name?",
    "What is the penalty for not paying minimum wages?",
    "Can a commercial dispute with a client be settled through arbitration?",
    "What is the capital of France?",   # out-of-scope control
]

eval_results = []
for q in EVAL_QUERIES:
    try:
        s = legal_graph.invoke(new_state(q))
        if s["needs_clarification"]:
            outcome = "Asked clarification"
        elif s["citations"]:
            outcome = "Answered with citations"
        elif s["insufficient_evidence"]:
            outcome = "Abstained (insufficient evidence)"
        else:
            outcome = "Unexpected state"
        cited = "; ".join(
            f'{c["act_name"]} s.{c["section_number"]}' for c in s["citations"]
        ) or "-"
        eval_results.append((q, outcome, s["verified"], cited, s["answer"]))
    except Exception as e:
        eval_results.append((q, f"ERROR: {type(e).__name__}", False, "-", str(e)[:200]))

print("| # | Query | Outcome | Verified | Cited sections |")
print("|---|-------|---------|----------|----------------|")
for i, (q, outcome, ver, cited, _) in enumerate(eval_results, 1):
    print(f"| {i} | {q} | {outcome} | {ver} | {cited} |")

print("\n" + "=" * 60)
for i, (q, outcome, ver, cited, ans) in enumerate(eval_results, 1):
    print(f"\n[{i}] {q}\n{(ans or '')[:700]}")

In [ ]:
import gradio as gr

pending = {"query": None}   # holds the original question while we wait for a clarification answer

def chat_fn(message, history):
    message = (message or "").strip()
    if not message:
        return "Please type a question."

    # If the last turn was a clarification question, combine it with the user's reply
    query = f"{pending['query']} (Additional details: {message})" if pending["query"] else message
    pending["query"] = None

    try:
        s = legal_graph.invoke(new_state(query))
    except LLMUnavailableError:
        return "The language model is temporarily unavailable. Please try again in a minute."
    except Exception as e:
        return f"Something went wrong ({type(e).__name__}). Please try rephrasing your question."

    if s["needs_clarification"]:
        pending["query"] = query
        return s["answer"]

    reply = s["answer"]

    if s["citations"]:
        sources = "\n".join(
            f"- {c['act_name']}, Section {c['section_number']} ({c['section_title']})"
            for c in s["citations"]
        )
        reply += f"\n\n**Sources**\n{sources}"
        reply += ("\n\n*Citation check passed: the cited sections were retrieved. "
                  "This does not verify that every legal claim is correct.*")
    else:
        reply += "\n\n*No supporting sources were found for this question.*"

    return reply

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Indian Startup Legal Assistant (prototype)",
    description="General legal information from indexed Indian statutes. Not legal advice.",
    examples=[
        "Can my employer fire me while I am pregnant?",
        "What are the duties of directors of a company?",
        "Is an employer required to pay gratuity?",
    ],
)

demo.launch(debug=False)